# Qwen2.5-VL eval — clean Drive edition (BF16, fast)

Apples-to-apples vs Claude on the 90-image test set. All version pins
and speed optimizations baked in.

## One-time Drive setup

Before running, make sure your Drive has:
```
MyDrive/ttb_sft/
├── qwen2_5_vl_7b/
│   └── adapter/                    ← already there from training
└── eval/
    └── cola_sample.csv             ← upload from test/eval/data/cola_sample.csv
```

## How to run

1. Runtime → Change runtime type → **A100 + High-RAM** → Save
2. Runtime → Disconnect and delete runtime (clean VM)
3. Runtime → **Run all** ← 1st click. Installs deps, kernel auto-crashes (red banner = expected).
4. Runtime → **Run all** ← 2nd click. Drive permission dialog pops, accept it. Runs end-to-end (~10 min on A100-80GB).
5. `qwen_outputs.jsonl` auto-downloads to your Mac's `~/Downloads/`. Then locally: `make eval-replay-qwen`.

## Speed targets
- Model load: ~1 min (HF cache hits on 2nd+ runs)
- Per-image inference: **2-4 sec** (BF16 on A100-80GB; 4-bit was ~21 sec)
- Total extraction: **~5 min** for 90 images

## 1. Install dependencies (auto-restart after first run)

> Installs ~3 GB. Wait for the red "session crashed" banner, then Runtime → Run all AGAIN.

In [ ]:
# Verified-working stack for Qwen2.5-VL + LoRA inference:
#   transformers==4.51.3   has Qwen2.5-VL + the 4.49 config-regression fix
#   peft>=0.13             load the LoRA adapter (requires torchao>=0.16)
#   torchao>=0.16          required by peft; Colab's 0.10 is too old
#   accelerate>=0.30, bitsandbytes>=0.44, qwen-vl-utils, protobuf>=5.27
#   numpy>=2.0,<2.2        wheel ABI sweet spot
import importlib.util, subprocess, sys, os


def _have(mod): return importlib.util.find_spec(mod) is not None
def _pinned(mod, want):
    try: return __import__(mod).__version__ == want
    except Exception: return False
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split(".")
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception: return False
def _torchao_ok():
    try:
        import torchao
        major, minor = [int(x) for x in torchao.__version__.split(".")[:2]]
        return (major, minor) >= (0, 16)
    except Exception:
        return False


_required = ["peft", "accelerate", "PIL", "qwen_vl_utils"]
_ready = (
    _numpy_ok()
    and _pinned("transformers", "4.51.3")
    and _torchao_ok()
    and all(_have(m) for m in _required)
)

if _ready:
    import numpy, transformers, torchao
    print(f"✓ Set up — numpy {numpy.__version__}, transformers {transformers.__version__}, torchao {torchao.__version__}")
else:
    print("⏳ Installing dependencies (~3 min). Kernel will auto-restart at end.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers==4.51.3",
        "accelerate>=0.30",
        "peft>=0.13",
        "torchao>=0.16",          # peft needs >=0.16; Colab ships 0.10
        "bitsandbytes>=0.44",     # not used in bf16 path but harmless
        "qwen-vl-utils",
        "protobuf>=5.27",
        "pillow", "tqdm"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--force-reinstall", "--no-deps", "numpy>=2.0,<2.2"])
    print()
    print("=" * 70)
    print("✅ Install complete. Auto-restarting kernel so the new ABI takes effect.")
    print("   Wait for the 'session crashed' banner, then Runtime → Run all AGAIN.")
    print("=" * 70)
    os.kill(os.getpid(), 9)

## 2. Mount Drive + verify the adapter and CSV are in place

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT  = Path("/content/drive/MyDrive/ttb_sft")
ADAPTER_DIR = DRIVE_ROOT / "qwen2_5_vl_7b" / "adapter"
CSV_PATH    = DRIVE_ROOT / "eval" / "cola_sample.csv"
OUTPUT_DIR  = DRIVE_ROOT / "eval"

if not ADAPTER_DIR.exists() or not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"\nMissing LoRA adapter at {ADAPTER_DIR}\n"
        f"It should be there from the training notebook. If not, re-run training "
        f"or upload the adapter folder to that Drive path."
    )
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"\nMissing CSV at {CSV_PATH}\n"
        f"Upload test/eval/data/cola_sample.csv from your Mac into "
        f"MyDrive/ttb_sft/eval/ (create the eval/ folder if needed), then re-run."
    )

print(f"✓ Adapter: {ADAPTER_DIR}")
print(f"✓ CSV:     {CSV_PATH} ({CSV_PATH.stat().st_size:,} bytes)")
print(f"✓ Output:  {OUTPUT_DIR}")

## 3. Load Qwen2.5-VL in BF16 + apply LoRA + merge

We load in **BF16, not 4-bit**. The A100-80GB has plenty of VRAM and BF16 inference is ~4× faster than bnb 4-bit. First load downloads the ~16 GB base from HF (~5 min cold, instant on cache hit).

In [ ]:
import torch, time
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

print("Loading Qwen2.5-VL-7B-Instruct in BF16...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",   # 1.5-2x faster than eager, no flash-attn dep
)
print(f"Applying LoRA from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()

print("Merging LoRA into base (safe in BF16 — eliminates per-step LoRA overhead)...")
model = model.merge_and_unload()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

print(f"✓ Model ready. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

# Speed sanity check
from PIL import Image
img_test = Image.new("RGB", (448, 448), color="white")
msgs = [{"role": "user", "content": [{"type": "image", "image": img_test},
                                      {"type": "text", "text": "describe in 50 words"}]}]
text_test = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text_test], images=[img_test], return_tensors="pt").to(model.device)
with torch.no_grad():
    _ = model.generate(**inputs, max_new_tokens=10, do_sample=False, temperature=None, top_p=None,
                       pad_token_id=processor.tokenizer.eos_token_id)
t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100, do_sample=False, temperature=None, top_p=None,
                         pad_token_id=processor.tokenizer.eos_token_id)
tok_s = 100 / (time.time() - t0)
print(f"Speed: {tok_s:.0f} tok/s on small input (expect 50-80 on A100-80GB)")

## 4. Load CSV + download test images from the COLA Cloud CDN

In [ ]:
import csv, urllib.request, concurrent.futures

rows = list(csv.DictReader(open(CSV_PATH)))
print(f"Loaded {len(rows)} rows from {CSV_PATH.name}")

IMG_DIR = Path("/content/qwen_eval_images")
IMG_DIR.mkdir(exist_ok=True)
CDN = "https://dyuie4zgfxmt6.cloudfront.net/{}.webp"

def _dl(row):
    image_id = row["ttb_image_id"]
    dest = IMG_DIR / f"{image_id}.webp"
    if dest.exists() and dest.stat().st_size > 0:
        return row, True
    try:
        req = urllib.request.Request(CDN.format(image_id),
                                     headers={"User-Agent": "ttb-eval/0.1"})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return row, True
    except Exception as e:
        print(f"  download fail {image_id}: {e}")
        return row, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for row, ok in ex.map(_dl, rows):
        if ok:
            row["_image_path"] = str(IMG_DIR / f"{row['ttb_image_id']}.webp")
            downloaded.append(row)
print(f"✓ Downloaded {len(downloaded)} / {len(rows)} images")

## 5. Run Qwen extraction (resize images to cap visual-token count)

Pre-resizes each image to ≤640×640 area before processing. Without this,
Qwen2.5-VL generates 4000+ visual tokens per image, making attention
quadratic-slow. With the cap, it's ~400 visual tokens — plenty for OCR.

In [ ]:
import json, re, time, statistics
from PIL import Image
from collections import Counter
SYSTEM_PROMPT = (
    "You are a careful transcription assistant for U.S. TTB alcohol label review. "
    "Given ONE label panel image and the beverage type, READ what is printed and "
    "RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n"
    "CRITICAL — use ONLY these EXACT field names in the fields object (case-sensitive, "
    "preserve spaces and punctuation):\n"
    "  - 'Brand name'\n"
    "  - 'Class & type'\n"
    "  - 'Alcohol content'\n"
    "  - 'Net contents'\n"
    "  - 'Bottler name/address'\n"
    "  - 'Country of origin'  (imports only)\n\n"
    "Do NOT invent variants ('brand_name', 'Volume', 'Bottled_by', 'product_type', "
    "'location', etc). If a field is not clearly visible on this panel, OMIT it from "
    "the fields object entirely. Never guess; never substitute deposit codes "
    "(e.g. \"CA CRV\"), NOM IDs, or barcodes. Transcribe the Government Warning "
    "EXACTLY as printed (preserve case, punctuation, errors).\n\n"
    "For government_warning.casing_all_caps: TRUE if the literal text "
    "'GOVERNMENT WARNING' appears in ALL CAPS in the image; FALSE otherwise. "
    "For government_warning.body_bold: TRUE if the body text (after the heading) "
    "appears bold; FALSE otherwise (the body should NOT be bold per 27 CFR 16.22).\n\n"
    "Schema: {fields: {<field name>: {value, confidence}}, government_warning: "
    "{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, "
    "contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}"
)

MAX_PIXELS_HARD = 640 * 640  # ~400 visual tokens after Qwen2.5-VL patching

def _resize_for_speed(img):
    """Cap image area + snap dims to multiples of 28 (Qwen2.5-VL patch size)."""
    w, h = img.size
    if w * h > MAX_PIXELS_HARD:
        scale = (MAX_PIXELS_HARD / (w * h)) ** 0.5
        img = img.resize((max(int(w * scale), 28), max(int(h * scale), 28)), Image.LANCZOS)
    w, h = img.size
    w28, h28 = max((w // 28) * 28, 28), max((h // 28) * 28, 28)
    if (w28, h28) != (w, h):
        img = img.resize((w28, h28), Image.LANCZOS)
    return img


def _qwen_extract(image_path, beverage_type):
    """Returns (parsed_dict_or_None, raw_decoded_text)."""
    img = _resize_for_speed(Image.open(image_path).convert("RGB"))
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": f"Beverage type: {beverage_type}. Extract per the schema."},
        ]},
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    decoded = processor.batch_decode(
        out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )[0]
    s, e = decoded.find("{"), decoded.rfind("}")
    if s == -1 or e == -1:
        return None, decoded
    try:
        return json.loads(decoded[s:e+1]), decoded
    except json.JSONDecodeError:
        cleaned = re.sub(r",(\s*[}\]])", r"\1", decoded[s:e+1])
        try: return json.loads(cleaned), decoded
        except Exception: return None, decoded


# Sanity check on the first real image
print("Sanity check on first image (this also warms CUDA kernels)...")
t0 = time.time()
pred0, _ = _qwen_extract(downloaded[0]["_image_path"],
                          (downloaded[0].get("beverage_type") or "wine").lower())
print(f"  ✓ First image: {time.time()-t0:.2f}s (expect 3-8s including warmup; subsequent will be faster)")

# Full extraction
qwen_outputs_jsonl = []
n_parsed = n_unparseable = 0
latencies = []

print(f"\nExtracting on {len(downloaded)} images...")
for i, row in enumerate(downloaded, 1):
    bev = (row.get("beverage_type") or "wine").lower()
    t0 = time.time()
    pred, raw_text = _qwen_extract(row["_image_path"], bev)
    dt = time.time() - t0
    latencies.append(dt)
    qwen_outputs_jsonl.append({
        "image_filename": row.get("image_filename") or (row["ttb_image_id"] + ".webp"),
        "ttb_image_id":   row["ttb_image_id"],
        "beverage_type":  bev,
        "raw_output":     raw_text,
        "parsed_output":  pred,
        "latency_sec":    round(dt, 3),
    })
    if pred is None: n_unparseable += 1
    else:            n_parsed += 1
    if i % 10 == 0:
        print(f"  [{i}/{len(downloaded)}] parsed={n_parsed} unparseable={n_unparseable} "
              f"latency_mean={statistics.mean(latencies):.2f}s")

print()
print("=" * 70)
print(f"Done. Parsed: {n_parsed} / Unparseable: {n_unparseable}")
print(f"Latency mean / p95: {statistics.mean(latencies):.2f}s / "
      f"{sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print("=" * 70)

## 6. Save outputs to Drive AND auto-download to your Mac

In [ ]:
jsonl_path = OUTPUT_DIR / "qwen_outputs.jsonl"
with open(jsonl_path, "w") as f:
    for rec in qwen_outputs_jsonl:
        f.write(json.dumps(rec) + "\n")
print(f"✓ Saved to Drive: {jsonl_path} ({len(qwen_outputs_jsonl)} rows)")

from google.colab import files
files.download(str(jsonl_path))

print()
print("Next on your Mac:")
print("  mv ~/Downloads/qwen_outputs.jsonl <wherever>   # optional, default works")
print("  make eval-replay-qwen")
print("    → writes test/eval/qwen_report.json + prints side-by-side vs Claude.")